# Querying Previous Radio Surveys
* NVSS
* FIRST

In [11]:
import os
import numpy as np
from otter import Otter, DataFinder
from astropy.table import vstack
from astropy import units as u


def print_for_copy(df):
    '''
    Print a df so that I can copy it over to the spreadsheet
    '''

    for _, row in df.iterrows():
        
        s = ''
        for col in df.columns:
            s += str(row[col]) + '\t'

        print(s)

In [2]:
otterpath = "private_otter_data"
overwrite = False

if overwrite:
    if os.path.exists(otterpath):
        for file in glob.glob(os.path.join(otterpath, "*")):
            os.remove(file)
        os.rmdir(otterpath)


    private_data = Otter.from_csvs("ecle-metadata.csv", photfile="all-photometry.csv", local_outpath=otterpath)
else:
    private_data = Otter(datadir=otterpath)

In [3]:
ecles = private_data.query(classification="ECLE", query_private=True)
len(ecles)

24

In [12]:
nvss_res = {}
first_res = {}

for e in ecles:
    c = e.get_skycoord()
    data_finder = DataFinder(c.ra, c.dec, "deg", "deg", name=e.default_name)
    
    nvss = data_finder.query_nvss()
    first = data_finder.query_first(radius=5*u.arcsec)
    
    if len(nvss) > 0:
        nvss_res[e.default_name] = nvss
        
    if len(first) > 0:
        first_res[e.default_name] = first

In [9]:
data_finder.query_first?

Signature:
data_finder.query_first(
    radius: 'u.Quantity' = <Quantity 5. arcmin>,
    get_image: 'bool' = False,
    **kwargs,
) -> 'list'
Docstring:
Query the FIRST radio survey and return an astropy table of the flux density

This queries Table 2 from Ofek & Frail (2011); 2011ApJ...737...45O

Args:
    radius (u.Quantity) : An astropy Quantity with the image height/width
    get_image (bool) : If True, download and return a list of the associated
                       images too.
    **kwargs : any other arguments to pass to the astroquery.image_cutouts
               get_images method

Returns:
    Astropy table of the flux densities. If get_image is True, it also returns
    a list of FIRST radio survey images
File:      ~/astro-otter/otter/src/otter/io/data_finder.py
Type:      method

In [13]:
nvss_res

{}

In [14]:
first_res

{'SDSS_J0938': TableList with 1 tables:
 	'0:J/ApJ/737/45/table2' with 8 column(s) and 1 row(s) }

In [15]:
first_tab = []
for name, value in first_res.items():
    tab = value.values()[0]
    tab["OBJECT"] = name
    first_tab.append(tab)
    
first_tab = vstack(first_tab)
first_tab["UPPERLIMIT"] = [row["Sp"] < 3*row["e_Sp"] for row in first_tab]
first_tab["day"] = np.nan
first_tab["date_format"] = "jd"
first_tab["flux_unit"] = "mJy"
first_tab["reducer"] = "2011ApJ...737...45O"
first_tab["source"] = "2011ApJ...737...45O"
first_tab["filter"] = "L"
first_tab["filter_eff"] = 1.4
first_tab["filter_eff_units"] = "GHz"
first_tab["array_config"] = np.nan



cols = ["OBJECT", "day", "t", "date_format", "array_config", "filter", "filter_eff", "filter_eff_units", "Sp", "e_Sp", "flux_unit", "UPPERLIMIT", "reducer", "source"]
print_for_copy(first_tab[cols].to_pandas())

SDSS_J0938	nan	2451526.264	jd	nan	L	1.4	GHz	1.93	0.15000000596046448	mJy	False	2011ApJ...737...45O	2011ApJ...737...45O	
